***Импорт библиотек***

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

***Функция загрузки датасета***

In [2]:


def get_data_loaders(
    train_transform,
    test_transform,
    batch_size=32,
    train_path="MY_data/train",
    test_path="MY_data/test"
):
    train_data = datasets.ImageFolder(
        train_path,
        transform=train_transform
    )

    test_data = datasets.ImageFolder(
        test_path,
        transform=test_transform
    )

    train_loader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True
    )

    test_loader = DataLoader(
        test_data,
        batch_size=batch_size,
        shuffle=False
    )

    return train_data, test_data, train_loader, test_loader

***Функция обучения***

In [3]:

def train_model(model, train_loader, criterion, optimizer, device, epochs):
    model.to(device)

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

***Функция оценки***

In [4]:

def evaluate_model(model, test_loader, device):
    model.eval()

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')

    print("\nResults:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")

    return accuracy, precision, recall, f1

***Baseline transforms***

In [5]:
baseline_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_data_baseline, test_data_baseline, train_loader_baseline, test_loader_baseline = get_data_loaders(
    baseline_transform,
    baseline_transform
)

***Baseline ResNet из torchvision***

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(train_data_baseline.classes)

model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_model(model, train_loader_baseline, criterion, optimizer, device, epochs=5)
evaluate_model(model, test_loader_baseline, device)

/home/kirill/labAIlast/venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/kirill/labAIlast/venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/5, Loss: 0.9245
Epoch 2/5, Loss: 0.4723
Epoch 3/5, Loss: 0.3898
Epoch 4/5, Loss: 0.3762
Epoch 5/5, Loss: 0.2193

Results:
Accuracy: 0.6439
Precision: 0.6825
Recall: 0.6439
F1-score: 0.6502


(0.6439024390243903,
 0.6825238268712657,
 0.6439024390243903,
 0.6501932515324929)

***Гипотезы:***
1. Увелечение количества эпох(5->10) позволит модели лучше извлечь признаки изображений и повысить качество классификации.
2. Применение аугментации(RandomHorizontalFlip, RandomRotation, ColorJitter) данных улучшит обобщающую способность модели.
3. Уменьшение learning rate(0.001->0.0001) сделает обучение более стабильным

***Improved transforms***

In [7]:
train_aug_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor()
])

test_aug_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_data_improved, test_data_improved, train_loader_improved, test_loader_improved = get_data_loaders(
    train_aug_transform,
    test_aug_transform
)

***Improved  ResNet из torchvision***

In [8]:

num_classes = len(train_data_baseline.classes)

model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

train_model(model, train_loader_improved, criterion, optimizer, device, epochs=10)
evaluate_model(model, test_loader_improved, device)

/home/kirill/labAIlast/venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/kirill/labAIlast/venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/10, Loss: 0.7494
Epoch 2/10, Loss: 0.1907
Epoch 3/10, Loss: 0.1054
Epoch 4/10, Loss: 0.0794
Epoch 5/10, Loss: 0.0565
Epoch 6/10, Loss: 0.0416
Epoch 7/10, Loss: 0.0308
Epoch 8/10, Loss: 0.0380
Epoch 9/10, Loss: 0.0275
Epoch 10/10, Loss: 0.0220

Results:
Accuracy: 0.7229
Precision: 0.7421
Recall: 0.7229
F1-score: 0.7297


(0.7229268292682927,
 0.7420766397165188,
 0.7229268292682927,
 0.7297373748466316)

***Результаты:***

| Метрика   | Было  | Стало |
|------------|--------|--------|
| Accuracy   | 0.6439 | 0.7229 |
| Precision  | 0.6825 | 0.7421 |
| Recall     | 0.6439 | 0.7229 |
| F1-score   | 0.6502 | 0.7297 |

***Выводы:***

Метрики улучшились, гипотезы подтвердились.

***Имплементация алгоритма машинного обучения(CNN)***

In [9]:
class MyCNN(nn.Module):
    def __init__(self, num_classes):
        super(MyCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = MyCNN(num_classes)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_model(model, train_loader_baseline, criterion, optimizer, device, epochs=5)
evaluate_model(model, test_loader_baseline, device)

Epoch 1/5, Loss: 2.0955
Epoch 2/5, Loss: 1.6678
Epoch 3/5, Loss: 1.4978
Epoch 4/5, Loss: 1.3495
Epoch 5/5, Loss: 1.3031

Results:
Accuracy: 0.3980
Precision: 0.4311
Recall: 0.3980
F1-score: 0.3881


(0.3980487804878049,
 0.4310803400244296,
 0.3980487804878049,
 0.38809311862279094)

***Результаты:***

| Метрика   | torchvision  | MyCNN |
|------------|--------|--------|
| Accuracy   | 0.6439 | 0.3980 |
| Precision  | 0.6825 | 0.4311 |
| Recall     | 0.6439 | 0.3980 |
| F1-score   | 0.6502 | 0.3881 |

***Выводы:***

Метрики у алгоритма из torchvision оказались лучше, что обьясняется упрощенной реализацией MyCNN.

***Улучшение MyCNN***

In [10]:
model = MyCNN(num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

train_model(model, train_loader_improved, criterion, optimizer, device, epochs=10)
evaluate_model(model, test_loader_improved, device)

Epoch 1/10, Loss: 2.2305
Epoch 2/10, Loss: 1.8562
Epoch 3/10, Loss: 1.6887
Epoch 4/10, Loss: 1.5706
Epoch 5/10, Loss: 1.5005
Epoch 6/10, Loss: 1.4434
Epoch 7/10, Loss: 1.3779
Epoch 8/10, Loss: 1.3372
Epoch 9/10, Loss: 1.2668
Epoch 10/10, Loss: 1.2376

Results:
Accuracy: 0.4673
Precision: 0.4683
Recall: 0.4673
F1-score: 0.4556


(0.46731707317073173,
 0.46834193446176603,
 0.46731707317073173,
 0.4556440330333335)

***Результаты:***

| Метрика   | MyCNN  | torchvision |
|------------|--------|--------|
| Accuracy   | 0.4673 | 0.7229 |
| Precision  | 0.4683 | 0.7421 |
| Recall     | 0.4673 | 0.7229 |
| F1-score   | 0.4556 | 0.7297 |

***Выводы:***

После применения гипотез результаты у алгоритма из torchvision оказались лучше чем у собственной реализации. 

***Baseline Vision Transformer из torchvision***

In [11]:
model = models.vit_b_16(pretrained=True)

model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_model(model, train_loader_baseline, criterion, optimizer, device, epochs=5)
evaluate_model(model, test_loader_baseline, device)

/home/kirill/labAIlast/venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/kirill/labAIlast/venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_B_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_B_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /home/kirill/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100.0%


Epoch 1/5, Loss: 2.4289
Epoch 2/5, Loss: 2.3575
Epoch 3/5, Loss: 2.0337
Epoch 4/5, Loss: 1.7488
Epoch 5/5, Loss: 1.6497

Results:
Accuracy: 0.2449
Precision: 0.2730
Recall: 0.2449
F1-score: 0.2142


/home/kirill/labAIlast/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


(0.2448780487804878,
 0.2730229822373222,
 0.2448780487804878,
 0.21418682317778817)

***Improved Vision Transformer из torchvision***

In [12]:
model = models.vit_b_16(pretrained=True)

model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

train_model(model, train_loader_improved, criterion, optimizer, device, epochs=10)
evaluate_model(model, test_loader_improved, device)

/home/kirill/labAIlast/venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/kirill/labAIlast/venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_B_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_B_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/10, Loss: 0.5146
Epoch 2/10, Loss: 0.1162
Epoch 3/10, Loss: 0.0605
Epoch 4/10, Loss: 0.0635
Epoch 5/10, Loss: 0.0749
Epoch 6/10, Loss: 0.0551
Epoch 7/10, Loss: 0.0200
Epoch 8/10, Loss: 0.0352
Epoch 9/10, Loss: 0.0548
Epoch 10/10, Loss: 0.0539

Results:
Accuracy: 0.7278
Precision: 0.7416
Recall: 0.7278
F1-score: 0.7338


(0.7278048780487805,
 0.7416250656558621,
 0.7278048780487805,
 0.7338033690239227)

***Результаты:***

| Метрика   | Было  | Стало |
|------------|--------|--------|
| Accuracy   | 0.2449 | 0.7278 |
| Precision  | 0.2730 | 0.7416 |
| Recall     | 0.2449 | 0.7278 |
| F1-score   | 0.2142 | 0.7338 |

***Выводы:***

Метрики значительно улучшились, гипотезы подтвердились.

***Имплементация алгоритма машинного обучения(Vision Transformer)***

In [18]:


class MyViT(nn.Module):
    def __init__(
        self,
        image_size=224,
        patch_size=16,
        num_classes=10,
        embed_dim=128,
        num_heads=4,
        num_layers=2
    ):
        super(MyViT, self).__init__()

        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        patch_dim = 3 * patch_size * patch_size

        self.patch_embedding = nn.Linear(patch_dim, embed_dim)

        self.cls_token = nn.Parameter(
            torch.randn(1, 1, embed_dim)
        )

        self.position_embedding = nn.Parameter(
            torch.randn(1, self.num_patches + 1, embed_dim)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.classifier = nn.Linear(
            embed_dim,
            num_classes
        )

    def forward(self, x):
        batch_size = x.shape[0]

        patches = x.unfold(2, self.patch_size, self.patch_size)\
                   .unfold(3, self.patch_size, self.patch_size)

        patches = patches.contiguous().view(
            batch_size,
            -1,
            3 * self.patch_size * self.patch_size
        )

        x = self.patch_embedding(patches)

        cls_tokens = self.cls_token.expand(
            batch_size,
            -1,
            -1
        )

        x = torch.cat([cls_tokens, x], dim=1)

        x = x + self.position_embedding

        x = self.transformer(x)

        cls_output = x[:, 0]

        x = self.classifier(cls_output)

        return x

model = MyViT(image_size=224, patch_size=16, num_classes=num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_model(model, train_loader_baseline, criterion, optimizer, device, epochs=5)
evaluate_model(model, test_loader_baseline, device)

Epoch 1/5, Loss: 2.3852
Epoch 2/5, Loss: 2.2570
Epoch 3/5, Loss: 2.1609
Epoch 4/5, Loss: 2.0857
Epoch 5/5, Loss: 2.0429

Results:
Accuracy: 0.2429
Precision: 0.2536
Recall: 0.2429
F1-score: 0.1908


/home/kirill/labAIlast/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


(0.24292682926829268,
 0.2536353225863152,
 0.24292682926829268,
 0.19082573470849148)

***Результаты:***

| Метрика   | torchvision  | MyViT |
|------------|--------|--------|
| Accuracy   | 0.2449 | 0.2429 |
| Precision  | 0.2730 | 0.2536 |
| Recall     | 0.2449 | 0.2429 |
| F1-score   | 0.2142 | 0.1908 |

***Выводы:***

Метрики у собственной реализации несколько хуже, чем у алгоритма из torchvision, что обьясняется упрощенной реалзиацией MyViT

***Улучшение MyViT***

In [19]:
model = MyViT(image_size=224, patch_size=16, num_classes=num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

train_model(model, train_loader_improved, criterion, optimizer, device, epochs=10)
evaluate_model(model, test_loader_improved, device)

Epoch 1/10, Loss: 2.3374
Epoch 2/10, Loss: 2.3197
Epoch 3/10, Loss: 2.2807
Epoch 4/10, Loss: 2.1516
Epoch 5/10, Loss: 2.0479
Epoch 6/10, Loss: 1.9427
Epoch 7/10, Loss: 1.8908
Epoch 8/10, Loss: 1.8736
Epoch 9/10, Loss: 1.8259
Epoch 10/10, Loss: 1.8112

Results:
Accuracy: 0.2683
Precision: 0.3382
Recall: 0.2683
F1-score: 0.2601


(0.2682926829268293,
 0.3382408160441839,
 0.2682926829268293,
 0.2600553247450508)

***Результаты:***

| Метрика   | MyViT  | torchvision |
|------------|--------|--------|
| Accuracy   | 0.2683 | 0.7278 |
| Precision  | 0.3382 | 0.7416 |
| Recall     | 0.2683 | 0.7278 |
| F1-score   | 0.2601 | 0.7338 |

***Выводы:***

После применения гипотез метрики у собственной реализации оказались значительно хуже, чем у алгоритма из torchvision.